In [340]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langgraph.graph.message import BaseMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv()  # Load environment variables from .env file


True

In [ ]:
from langgraph.graph.message import add_messages

# state
class ChatHistory(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages] # add_messages is a reducer that add chatModel response to the messages array, instead of replacing it....!


In [342]:
chatModel = ChatOpenAI(model_name="gpt-4o", temperature=0.7, max_tokens=100)

In [343]:
stateGraph = StateGraph(ChatHistory)

In [344]:
def chat_with_bot(state: ChatHistory): 

    print("state", state)
    # Get the last message from the user
    last_message = state['messages'][-1] if state['messages'] else None

    # # Generate a response using the chat model
    if last_message:
        response = chatModel.invoke([last_message])

    # print(last_message)

    return{ "messages":  [response] } # Append the response to the chat history

In [345]:
stateGraph.add_node('chat_with_bot', chat_with_bot)
stateGraph.add_edge(START, 'chat_with_bot')
stateGraph.add_edge('chat_with_bot', END)

In [346]:
compiledStateGraph = stateGraph.compile()


In [347]:
response1 =compiledStateGraph.invoke({"messages": [HumanMessage(content="1 + 1?")]})  # Start the conversation with an empty message list
response2 = compiledStateGraph.invoke({"messages": [HumanMessage(content="1 + 2?")]})  # Start the conversation with an empty message list

# here every invoke clears the state of the graph, so the second invoke does not have access to the first message.

print("response1: ", response1)
print("response2: ", response2)

state {'messages': [HumanMessage(content='1 + 1?', additional_kwargs={}, response_metadata={}, id='c41df9ad-7f46-41d3-ba0c-2330aca9e48f')]}
state {'messages': [HumanMessage(content='1 + 2?', additional_kwargs={}, response_metadata={}, id='ee65bbd6-2e52-4a25-8f32-9fd343e7716f')]}
response1:  {'messages': [HumanMessage(content='1 + 1?', additional_kwargs={}, response_metadata={}, id='c41df9ad-7f46-41d3-ba0c-2330aca9e48f'), AIMessage(content='1 + 1 equals 2.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 12, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2fe17714a7', 'id': 'chatcmpl-E9qPg9c7J8l1rE0ZNYiPJNMLCO43v', 'service_tier': '

In [ ]:
# persistent storage in RAM: 
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

stateGraph = StateGraph(ChatHistory)
stateGraph.add_node('chat_with_bot', chat_with_bot)
stateGraph.add_edge(START, 'chat_with_bot')
stateGraph.add_edge('chat_with_bot', END)

workflow = stateGraph.compile(checkpointer=checkpointer) # passing the checkpointer instance

In [ ]:
config1 = {"configurable": {"thread_id": "1"}} # thread
workflow.invoke({"messages": [HumanMessage("I am Sriharsha")]}, config= config1)

state {'messages': [HumanMessage(content='I am Sriharsha', additional_kwargs={}, response_metadata={}, id='b6feb323-3e87-4d20-8bcf-d2123f79d2cc')]}


{'messages': [HumanMessage(content='I am Sriharsha', additional_kwargs={}, response_metadata={}, id='b6feb323-3e87-4d20-8bcf-d2123f79d2cc'),
  AIMessage(content='Hello, Sriharsha! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 12, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f73b165a23', 'id': 'chatcmpl-E9qPpAMK9XgQktLBNuyrJskBa8Fyz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fd6cc-5b02-79e0-915a-14001421875c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 13, 'total_tokens': 25, 'input_token_details': {'audi

In [350]:
workflow.invoke({"messages": [HumanMessage("who am I")]}, config= config1)

state {'messages': [HumanMessage(content='I am Sriharsha', additional_kwargs={}, response_metadata={}, id='b6feb323-3e87-4d20-8bcf-d2123f79d2cc'), AIMessage(content='Hello, Sriharsha! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 12, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f73b165a23', 'id': 'chatcmpl-E9qPpAMK9XgQktLBNuyrJskBa8Fyz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fd6cc-5b02-79e0-915a-14001421875c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 13, 'total_tokens': 25, 'input_token_details': {'

{'messages': [HumanMessage(content='I am Sriharsha', additional_kwargs={}, response_metadata={}, id='b6feb323-3e87-4d20-8bcf-d2123f79d2cc'),
  AIMessage(content='Hello, Sriharsha! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 12, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_f73b165a23', 'id': 'chatcmpl-E9qPpAMK9XgQktLBNuyrJskBa8Fyz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fd6cc-5b02-79e0-915a-14001421875c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 13, 'total_tokens': 25, 'input_token_details': {'audi